does risk based pricing actually compensate for default risk, and would declining high risk segments outright improve profitability?


In [1]:
import pandas as pd

df = pd.read_csv("data/processed/lending_club_clean.csv")
LGD = 0.60  # unsecured consumer credit, industry-standard planning assumption -- stated explicitly, tested in Cell 4

In [2]:
df["risk_tier"] = pd.qcut(df["int_rate"], 4, labels=["A: Lowest rate", "B", "C", "D: Highest rate"])

tiers = df.groupby("risk_tier", observed=True).agg(
    n=("loan_amnt", "count"), avg_loan=("loan_amnt", "mean"),
    default_rate=("bad_loan", "mean"), avg_rate=("int_rate", "mean"), avg_term=("term_months", "mean"))

tiers["expected_loss"] = tiers.default_rate * LGD * tiers.avg_loan
tiers["expected_interest"] = tiers.avg_loan * (tiers.avg_rate / 100) * (tiers.avg_term / 12)
tiers["net_expected_value"] = (1 - tiers.default_rate) * tiers.expected_interest - tiers.expected_loss
tiers.round(2)

,n,avg_loan,default_rate,avg_rate,avg_term,expected_loss,expected_interest,net_expected_value
risk_tier,,,,,,,,
A: Lowest rate,329441,13733.62,0.08,7.70,37.24,644.37,3283.17,2382.05
B,323387,13235.67,0.16,11.41,39.52,1241.01,4972.94,2954.81
C,325373,14146.71,0.23,14.26,42.57,1913.28,7156.02,3629.71
D: Highest rate,325437,16552.26,0.34,19.72,47.90,3419.31,13025.34,5121.49


In [3]:
df["dti_band"] = pd.cut(df["dti"], [0, 10, 20, 30, 100], labels=["0-10", "10-20", "20-30", "30+"])

print(df.groupby("purpose")["bad_loan"].mean().sort_values(ascending=False).round(3))
print(df.groupby("dti_band", observed=True)["bad_loan"].mean().round(3))

purpose
small_business        0.297
renewable_energy      0.237
moving                0.234
medical               0.219
house                 0.217
debt_consolidation    0.213
other                 0.212
vacation              0.192
major_purchase        0.187
home_improvement      0.179
educational           0.172
credit_card           0.170
car                   0.146
wedding               0.122
Name: bad_loan, dtype: float64
dti_band
0-10     0.149
10-20    0.179
20-30    0.232
30+      0.295
Name: bad_loan, dtype: float64


In [6]:
#policy sim

def net_impact(mask):
    flagged = df[mask]
    realized_value = (flagged.total_pymnt + flagged.recoveries - flagged.loan_amnt).sum()
    return -realized_value  # declining removes this realized value entirely

rules = {
    "dti>25 & revol_util>80": (df.dti > 25) & (df.revol_util > 80),
    "dti>30": df.dti > 30,
    "revol_util>90": df.revol_util > 90,
    "small_business & dti>20": (df.purpose == "small_business") & (df.dti > 20),
}

results = {}
for label, mask in rules.items():
    flagged = df[mask]
    net = net_impact(mask)
    results[label] = net
    print(f"{label}: n={len(flagged):,} ({len(flagged)/len(df)*100:.1f}%) "
          f"default_rate={flagged.bad_loan.mean()*100:.1f}% net=${net:,.0f} "
          f"(${net/len(flagged):,.0f} per flagged loan)")

worst_label = min(results, key=results.get)
profitable = [k for k, v in results.items() if v > 0]
print(f"\nWorst rule: {worst_label} (net=${results[worst_label]:,.0f})")
print(f"{len(profitable)} of {len(rules)} rules would have been profitable to decline "
      f"based on realized outcomes: {', '.join(profitable)}")

dti>25 & revol_util>80: n=49,377 (3.8%) default_rate=28.7% net=$-27,094,858 ($-549 per flagged loan)
dti>30: n=123,487 (9.5%) default_rate=29.5% net=$1,545,839 ($13 per flagged loan)
revol_util>90: n=78,266 (6.0%) default_rate=23.5% net=$-75,167,757 ($-960 per flagged loan)
small_business & dti>20: n=3,845 (0.3%) default_rate=35.2% net=$2,067,328 ($538 per flagged loan)

Worst rule: revol_util>90 (net=$-75,167,757)
2 of 4 rules would have been profitable to decline based on realized outcomes: dti>30, small_business & dti>20


In [7]:
tiers.reset_index().to_csv("data/processed/risk_tiers.csv", index=False)

rule_rows = []
for label, mask in rules.items():
    flagged = df[mask]
    rule_rows.append({
        "rule": label, "n": len(flagged),
        "default_rate": flagged.bad_loan.mean() * 100,
        "net_impact": net_impact(mask),
    })
pd.DataFrame(rule_rows).to_csv("data/processed/underwriting_rules.csv", index=False)

purpose_summary = df.groupby("purpose").agg(
    default_rate=("bad_loan", "mean"), exposure=("loan_amnt", "sum")).reset_index()
purpose_summary["default_rate"] *= 100
purpose_summary.to_csv("data/processed/purpose_risk.csv", index=False)

dti_summary = df.groupby("dti_band", observed=True)["bad_loan"].mean().reset_index()
dti_summary.columns = ["dti_band", "default_rate"]
dti_summary["default_rate"] *= 100
dti_summary.to_csv("data/processed/dti_risk.csv", index=False)